# Training llama models from scratch

Import all needed libraries:

In [ ]:
import torch
import pandas as pd
from random import sample
from pathlib import Path
from tqdm import tqdm 

from tokenizers import (
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    Tokenizer)

from tokenizers.normalizers import Lowercase, Strip, StripAccents, NFD

from transformers import (
    AutoTokenizer, 
    PreTrainedTokenizerFast, 
    set_seed, 
    Trainer, 
    TrainingArguments, 
    DataCollatorForLanguageModeling, 
    LlamaForCausalLM, 
    LlamaConfig)

from datasets import load_dataset

### Paths

Set paths to training data, eval data, and model directory:

In [ ]:
base = r"C:/Users/ppk_2/Desktop/University/SummerSem-25/Neural Networks for NLP/Assignments/Main"
clean = f"{base}/clean"
model_root = f"{base}/models"

training_files = [f"{clean}/corpus_A.txt"]          # for Model A
eval_files     = [f"{clean}/corpus_A.txt"]          # or make a small held-out slice
model_path     = f"{model_root}/babyA/"

### Tokenizer

Initialize with BPE:

In [ ]:
# Train-once, shared tokenizer directory (used by BOTH models A & B)
tokenizer_dir = f"{base}/models/tokenizer"

In [ ]:
# --- Train the tokenizer ONCE on the union corpus ---
from tokenizers import Tokenizer, decoders, normalizers, pre_tokenizers, processors, models, trainers
from tokenizers.normalizers import NFD, Lowercase, Strip
from transformers import PreTrainedTokenizerFast, AutoTokenizer
from datasets import load_dataset
import os

os.makedirs(tokenizer_dir, exist_ok=True)

In [ ]:
# 1) Build tokenizer with ByteLevel BPE
tokenizer = Tokenizer(models.BPE())
tokenizer.normalizer = normalizers.Sequence([NFD(), Lowercase(), Strip()])  # lowercase optional; aligns with plan
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)   # GPT2-style safety with leading spaces

Normalizer that sets everything to normal unicode, lowercase, and strips white spaces and accents

(explanations here: https://huggingface.co/docs/tokenizers/components)

In [ ]:
# 2) Train on the union corpus (shared vocab for A & B)
trainer = trainers.BpeTrainer(
    vocab_size=8000,                              # plan suggests ~8k; change to 16000 if you prefer
    special_tokens=["<|endoftext|>", "<pad>"]     # eos + pad (we'll set bos=eos for causal LM)
)
tokenizer.train(files=[f"{clean}/corpus_union.txt"], trainer=trainer)

In [ ]:
# 3) Post-processing & decoder (ByteLevel)
tokenizer.post_processor = processors.ByteLevel(trim_offsets=True)
tokenizer.decoder = decoders.ByteLevel()

In [ ]:
# 4) Wrap as HF tokenizer and save to shared dir
wrapped_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    bos_token="<|endoftext|>",
    eos_token="<|endoftext|>",
    pad_token="<pad>",
)
wrapped_tokenizer.save_pretrained(tokenizer_dir)

In [ ]:
# 5) Load the shared tokenizer (for training either A or B)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)
tokenizer.pad_token = tokenizer.eos_token   # common trick for causal LM
context_length = 128                        # set this now; we’ll match model config to 128 later

In [ ]:
# 6) Build datasets (text) for the selected model's files
raw_datasets = load_dataset(
    "text",
    data_files={
        "train": training_files,
        "validation": eval_files
    }
)

In [ ]:
# 7) Tokenize, then concatenate & chunk to fixed-length blocks (best for causal LM)
def tokenize_function(examples):
    return tokenizer(examples["text"], return_attention_mask=False, return_token_type_ids=False,)

tokenized = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)

# Concatenate all input_ids, then split into blocks of context_length
def group_texts(examples):
    # Concatenate
    concatenated = sum(examples["input_ids"], [])
    # Drop remainder to make full blocks
    total_len = (len(concatenated) // context_length) * context_length
    concatenated = concatenated[:total_len]
    # Split by chunks
    result = {
        "input_ids": [concatenated[i:i+context_length] for i in range(0, total_len, context_length)]
    }
    return result

tokenized_datasets = tokenized.map(group_texts, batched=True)

In [ ]:
tokenized_datasets

Initiate new Llama with config as wished:

In [ ]:
len(tokenizer)

In [ ]:
import os, torch
from transformers import (
    LlamaConfig, LlamaForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments, Trainer, set_seed
)
import pandas as pd

In [ ]:
# --- Dirs (assumes model_path already set earlier) ---
os.makedirs(f"{model_path}/logs", exist_ok=True)
os.makedirs(f"{model_path}/final", exist_ok=True)

In [ ]:
# --- Match model to tokenizer/context length ---
# Make sure this equals the context_length you used when chunking (128)
max_ctx = 128

config = LlamaConfig(
    vocab_size=len(tokenizer),
    hidden_size=128,
    num_hidden_layers=4,
    intermediate_size=128,
    num_attention_heads=4,
    bos_token_id=tokenizer.convert_tokens_to_ids("<|endoftext|>"),
    eos_token_id=tokenizer.convert_tokens_to_ids("<|endoftext|>"),
    pad_token_id=tokenizer.convert_tokens_to_ids("<pad>"),
    max_position_embeddings=max_ctx
)

Set seed for weight initialization:

In [ ]:
set_seed(42)

New model object:

In [ ]:
model = LlamaForCausalLM(config)
model.gradient_checkpointing_enable() # Optional: reduce VRAM at slight compute cost

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
# --- GPU / mixed precision setup ---
use_cuda = torch.cuda.is_available()
bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_ok = torch.cuda.is_available()  # most CUDA GPUs support fp16

Check out param size:

In [ ]:
# Small speedup on Ampere+ for matmul kernels
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# Choose precision flags
fp16_flag = False
bf16_flag = False
if use_cuda:
    if bf16_ok:
        bf16_flag = True
    else:
        fp16_flag = True

Set training parameters:

In [ ]:
training_args = TrainingArguments(
    output_dir=model_path,
    overwrite_output_dir=True,
    save_strategy = "epoch", # saves after every epoch
    #save_strategy = "steps",
    #save_steps = 0.1, # if below zero, then saves after every (n*100)% of training steps
    save_total_limit=2,  # set to zero to avoid saving
    eval_strategy = "epoch",
    #eval_steps = 0.1,
    num_train_epochs= 25,
    #max_steps = 1,
    gradient_accumulation_steps=8,
    per_device_train_batch_size=16,
    warmup_steps=200,
    lr_scheduler_type="cosine",
    learning_rate=3e-4, # normal: 5e-4
    logging_steps=10,
    fp16=fp16_flag, ## only on CUDA
    bf16=bf16_flag,              # True on Ampere+/newer if supported
    #load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    #use_mps_device=True, ## only on apple silicon
    #use_cpu = True,
    report_to=["none"],

)

Initialize trainer object:

In [ ]:
trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)

Train:

In [ ]:
train_result = trainer.train()
train_result

Save logs of losses:

In [ ]:
# --- Save logs & model ---
df = pd.DataFrame(trainer.state.log_history)
df.to_csv(f"{model_path}/logs/losses.csv", index=False)

Save final model

In [ ]:
trainer.save_model(f"{model_path}/final/")
tokenizer.save_pretrained(f"{model_path}/final/")  # keep tokenizer alongside weights

In [ ]:
print(f"model num parameters = {model.num_parameters()}")
print("CUDA used:", use_cuda, "| bf16:", bf16_flag, "| fp16:", fp16_flag)

### Test trained model on text generation

With hf pipelines:

In [ ]:
import torch, os
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# --- Paths ---
base = r"C:/Users/ppk_2/Desktop/University/SummerSem-25/Neural Networks for NLP/Assignments/Main"
tok_dir = f"{base}/models/tokenizer"             # shared tokenizer you saved earlier
model_path = f"{base}/models/babyA/final"        # <-- switch to babyB/final to test Model B

In [ ]:
# --- Load tokenizer & model on GPU if available ---
tokenizer = AutoTokenizer.from_pretrained(tok_dir)
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) \
    else (torch.float16 if torch.cuda.is_available() else torch.float32)

model = AutoModelForCausalLM.from_pretrained(model_path, dtype=dtype)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
# --- Build pipeline ---
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if device == "cuda" else -1
)

In [ ]:
# --- Prompts (feel free to edit) ---
prompts = [
    "Once upon a time in a small robotics lab,",
    "In natural language processing, a language model learns to",
    "The quickest way to evaluate grammatical preferences is to",
]

In [ ]:
# --- Sampling params (stable defaults) ---
gen_kwargs = dict(
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.1,
    num_return_sequences=3,
    max_new_tokens=80,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)


# --- Generate & print ---
for p in prompts:
    print(f"\n=== PROMPT: {p!r} ===")
    outs = pipe(p, **gen_kwargs)
    for i, o in enumerate(outs, 1):
        print(f"[{i}] {o['generated_text']}")

# Minicons


In [ ]:
# ===== BLiMP minimal-pair evaluation for Model A =====
import os, math, random, torch, pandas as pd
from datasets import get_dataset_config_names, load_dataset
from minicons import scorer
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"


In [ ]:
# --- Paths (edit only if your base changed) ---
base = r"C:/Users/ppk_2/Desktop/University/SummerSem-25/Neural Networks for NLP/Assignments/Main"
tok_dir = f"{base}/models/tokenizer"              # shared tokenizer dir (not strictly needed if tokenizer saved with model)
modelA_dir = f"{base}/models/babyA/final"         # Model A (already trained)
eval_dir = f"{base}/eval"
os.makedirs(eval_dir, exist_ok=True)

In [ ]:
# --- Minicons scorer on GPU if available ---
device = "cuda" if torch.cuda.is_available() else "cpu"
LM = scorer.IncrementalLMScorer(modelA_dir, device=device)  # loads model & tokenizer from this folder


In [ ]:
# --- Pick BLiMP phenomena and sample ~1000 pairs total ---
# We’ll randomly sample across many of the 67 BLiMP configs (each has 1000 pairs).
# This avoids hardcoding specific names and guarantees valid configs.
random.seed(42)
configs = get_dataset_config_names("nyu-mll/blimp")  # list of phenomena/configs from HF
random.shuffle(configs)

target_total = 1000
per_cfg_target = 100   # ~10 configs x 100 ≈ 1000 pairs
chosen = []
pairs = []

for cfg in configs:
    if len(chosen) >= 10 and len(pairs) >= target_total:
        break
    ds = load_dataset("nyu-mll/blimp", cfg, split="train")  # each row has sentence_good/sentence_bad
    n = min(per_cfg_target, len(ds))
    samp = ds.shuffle(seed=42).select(range(n))
    for ex in samp:
        pairs.append((cfg, ex["sentence_good"], ex["sentence_bad"]))
    chosen.append(cfg)

print(f"Using {len(chosen)} BLiMP phenomena, total pairs: {len(pairs)}")

# --- Score sentences in batches for speed ---
def batched_sequence_score(model_scorer, sentences, batch_size=32):
    out = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        try:
            scores = model_scorer.sequence_score(batch, reduction=sum)   # callable, preferred
        except TypeError:
            scores = model_scorer.sequence_score(batch, reduction="sum") # fallback for other versions
        # ensure Python floats on CPU
        for s in scores:
            if isinstance(s, torch.Tensor):
                out.append(s.detach().cpu().item())
            else:
                out.append(float(s))
    return out

phen, goods, bads = zip(*pairs)
good_scores = batched_sequence_score(LM, list(goods), batch_size=32)
bad_scores  = batched_sequence_score(LM, list(bads),  batch_size=32)

# --- Compute wins (higher logP wins) and build DataFrame ---
wins = [int(g > b) for g, b in zip(good_scores, bad_scores)]
df = pd.DataFrame({
    "phenomenon": phen,
    "sentence_good": goods,
    "sentence_bad": bads,
    "logp_good": good_scores,
    "logp_bad":  bad_scores,
    "correct": wins
})

# --- Per-phenomenon and overall accuracy ---
per_phen = (df.groupby("phenomenon")["correct"].mean().reset_index()
              .rename(columns={"correct": "accuracy"}))
per_phen = per_phen.sort_values("accuracy", ascending=False)

overall = df["correct"].mean()
print(f"\nModel A — BLiMP overall accuracy: {overall*100:.1f}%")
display(per_phen.head(10))

In [ ]:
# --- Save CSVs ---
eval_dir = f"{base}/eval"
os.makedirs(eval_dir, exist_ok=True)
df.to_csv(f"{eval_dir}/babyA_blimp_pairs_scores.csv", index=False)
per_phen.to_csv(f"{eval_dir}/babyA_blimp_summary.csv", index=False)
print(f"Saved: {eval_dir}/babyA_blimp_pairs_scores.csv and babyA_blimp_summary.csv")

# Model A vs B


In [ ]:
import pandas as pd, os

base = r"C:/Users/ppk_2/Desktop/University/SummerSem-25/Neural Networks for NLP/Assignments/Main"
eval_dir = f"{base}/eval"

# overall
a_pairs = pd.read_csv(f"{eval_dir}/babyA_blimp_pairs_scores.csv")
b_pairs = pd.read_csv(f"{eval_dir}/babyB_blimp_pairs_scores.csv")
overall_A = a_pairs["correct"].mean()*100
overall_B = b_pairs["correct"].mean()*100
print(f"Overall — A: {overall_A:.1f}% | B: {overall_B:.1f}% | Δ: {overall_B-overall_A:.1f} pp")

# per-phenomenon
a = pd.read_csv(f"{eval_dir}/babyA_blimp_summary.csv").rename(columns={"accuracy":"accuracy_A"})
b = pd.read_csv(f"{eval_dir}/babyB_blimp_summary.csv").rename(columns={"accuracy":"accuracy_B"})
m = a.merge(b, on="phenomenon", how="outer")
m["delta_pct"] = (m["accuracy_B"] - m["accuracy_A"])*100
m = m.sort_values("delta_pct", ascending=False)
display(m.head(10))
display(m.tail(10))
m.to_csv(f"{eval_dir}/A_vs_B_blimp_summary.csv", index=False)


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# --- EDIT THESE TWO PATHS ---
base = r"C:/Users/ppk_2/Desktop/University/SummerSem-25/Neural Networks for NLP/Assignments/Main"
path_A = f"{base}/models/babyA/final/logs/losses.csv"
path_B = f"{base}/models/babyB/final/logs/losses.csv"
out_path = f"{base}/eval/loss_curves_babyA_vs_babyB.png"
os.makedirs(os.path.dirname(out_path), exist_ok=True)

def prepare_curves(csv_path, ewma_alpha=0.1):
    """
    Reads a Hugging Face Trainer log_history CSV and returns:
      - tr: DataFrame with columns [step, loss, loss_smooth]
      - ev: DataFrame with columns [epoch, eval_loss]
    Handles missing 'step'/'epoch' robustly.
    """
    df = pd.read_csv(csv_path)
    # Training rows: have 'loss'
    tr = df[df.get("loss").notna() if "loss" in df else []].copy()
    if not tr.empty:
        if "step" not in tr or tr["step"].isna().all():
            tr["step"] = range(1, len(tr) + 1)
        tr = tr[["step", "loss"]].dropna().sort_values("step")
        tr["loss_smooth"] = tr["loss"].ewm(alpha=ewma_alpha).mean()
    else:
        tr = pd.DataFrame(columns=["step", "loss", "loss_smooth"])

    # Eval rows: have 'eval_loss'
    ev = df[df.get("eval_loss").notna() if "eval_loss" in df else []].copy()
    if not ev.empty:
        if "epoch" not in ev or ev["epoch"].isna().all():
            ev["epoch"] = range(1, len(ev) + 1)
        ev = ev[["epoch", "eval_loss"]].dropna().sort_values("epoch")
    else:
        ev = pd.DataFrame(columns=["epoch", "eval_loss"])

    return tr, ev

trA, evA = prepare_curves(path_A, ewma_alpha=0.1)
trB, evB = prepare_curves(path_B, ewma_alpha=0.1)

# Map eval epochs onto the step axis for a single-axes plot (approximate)
def epoch_to_steps(ev_df, tr_df):
    if ev_df.empty or tr_df.empty:
        return ev_df.assign(step_approx=pd.Series(dtype=float))
    max_step = tr_df["step"].max()
    max_epoch = max(ev_df["epoch"].max(), 1)
    step_approx = ev_df["epoch"] * (max_step / max_epoch)
    return ev_df.assign(step_approx=step_approx)

evA_ = epoch_to_steps(evA, trA)
evB_ = epoch_to_steps(evB, trB)

# --- Plot (single axes, no custom colors) ---
plt.figure(figsize=(9, 5))
if not trA.empty:
    plt.plot(trA["step"], trA["loss_smooth"], label="BabyA train (smoothed)")
    plt.plot(trA["step"], trA["loss"], linestyle="--", alpha=0.25, label="BabyA train (raw)")
if not trB.empty:
    plt.plot(trB["step"], trB["loss_smooth"], label="BabyB train (smoothed)")
    plt.plot(trB["step"], trB["loss"], linestyle="--", alpha=0.25, label="BabyB train (raw)")

if not evA_.empty:
    plt.scatter(evA_["step_approx"], evA_["eval_loss"], s=24, marker="o", label="BabyA eval")
if not evB_.empty:
    plt.scatter(evB_["step_approx"], evB_["eval_loss"], s=24, marker="x", label="BabyB eval")

plt.title("Training & Eval Loss — BabyA vs BabyB")
plt.xlabel("Training steps (approx)")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(out_path, dpi=150)
plt.show()
print("Saved plot to:", out_path)


In [ ]:
import numpy as np

def add_final_eval_annotations(ax, label, ev_df):
    if ev_df.empty:
        return
    final = ev_df.sort_values("epoch").iloc[-1]
    ppl = np.exp(final["eval_loss"])
    ax.annotate(f"{label}: eval {final['eval_loss']:.2f} (ppl {ppl:.1f})",
                xy=(final.get("step_approx", final["epoch"]), final["eval_loss"]),
                xytext=(10, 12), textcoords="offset points", fontsize=9)

# After you’ve prepared trA, evA, trB, evB and plotted:
add_final_eval_annotations(plt.gca(), "BabyA", evA_)
add_final_eval_annotations(plt.gca(), "BabyB", evB_)
plt.tight_layout()
plt.savefig(out_path, dpi=150)
